In [1]:
##### Calculates country-level and global percentile rasters for relative deprivation
# (population-weighted, based on the CIESIN GRDI relative deprivation index)

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from pathlib import Path
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt
from rasterio.warp import Resampling
from rasterio import warp as rio_warp

In [2]:
##### Load data

cd = Path.cwd().parent

# country geographies
countries = gpd.read_file("/Users/carinamanitius/Documents/Data/Admin_Boundaries/gadm_410-levels.gpkg", layer="ADM_0",)

# reference raster (defines the analysis grid + land/non-land mask)
ref_path_reprojected = f"{cd}/Results/Raster_model/reprojected/terrain_slope.tif"

# source relative deprivation raster
relative_deprevation = f"{cd}/Data/Raw/Predictors/Relative_deprivation_CEISIN/povmap-grdi-v1.tif"
relative_deprevation_path_reprojected = (f"{cd}/Data/Raw/Predictors/Relative_deprivation_CEISIN/relative_deprevation_resampled.tif")

# population raster used to weight the CDF for this analysis 
pop_bondarenko_path = (f"{cd}/Data/Raw/Predictors/Population_Bondarenko/total_pop_2020_resampled_reprojected.tif")

# outputs
rd_percentiles_reprojected = f"{cd}/Data/Clean/GDP_percentiles/relative_deprevation_percentiles_reprojected.tif"
rd_global_percentiles_reprojected = (f"{cd}/Data/Clean/GDP_percentiles/global_relative_deprevation_percentiles_reprojected.tif")

In [3]:
##### Step 0: resample source RD raster onto the reference grid (average, gap-filled)

def resample_avg_to_ref(in_path, ref_path, out_path, band=1, atol=1e-9):
    with rasterio.open(ref_path) as ref:
        dst_crs, dst_transform, dst_shape = ref.crs, ref.transform, ref.shape
        ref_masked = ref.read(1, masked=True)
        ref_nodata_mask = np.ma.getmaskarray(ref_masked)
        ref_res = ref.res

    with rasterio.open(in_path) as src:
        src_res = src.res
        grids_match = (
            src.crs == dst_crs
            and src.shape == dst_shape
            and src.transform.almost_equals(dst_transform, precision=atol)
        )

        if grids_match:
            arr = src.read(band, masked=True).filled(np.nan).astype(np.float32)
            out_meta = src.meta.copy()
            out_meta.update(dtype="float32", count=1, nodata=np.nan)
            with rasterio.open(out_path, "w", **out_meta) as dst:
                dst.write(arr, 1)
            print(f"{in_path} (band {band}): grid already matches ref_path — saved without resampling.")
            return arr

        if not (ref_res[0] >= src_res[0] - atol and ref_res[1] >= src_res[1] - atol):
            raise ValueError(
                f"ref_path resolution {ref_res} is not coarser than in_path "
                f"resolution {src_res} — averaging resample requires ref_path coarser."
            )

        src_nodata = src.nodatavals[band - 1]
        src_valid_mean = src.read(band, masked=True).mean()

        dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
        rio_warp.reproject(
            source=rasterio.band(src, band),
            destination=dst_array,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src_nodata,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            dst_nodata=np.nan,
            resampling=Resampling.average,
        )

    # fill gaps (valid ref land, no source data) with nearest valid pixel
    gap_mask = np.isnan(dst_array) & ~ref_nodata_mask
    if gap_mask.any():
        valid_mask = ~np.isnan(dst_array)
        _, (nearest_i, nearest_j) = distance_transform_edt(
            ~valid_mask, return_distances=True, return_indices=True
        )
        dst_array[gap_mask] = dst_array[nearest_i[gap_mask], nearest_j[gap_mask]]

    dst_array[ref_nodata_mask] = np.nan

    out_meta = dict(
        driver="GTiff", height=dst_shape[0], width=dst_shape[1], count=1,
        dtype="float32", crs=dst_crs, transform=dst_transform, nodata=np.nan,
    )
    with rasterio.open(out_path, "w", **out_meta) as dst:
        dst.write(dst_array.astype("float32"), 1)

    mean_after = np.nanmean(dst_array)
    print(
        f"{in_path} (band {band}): before_mean={src_valid_mean:.4f}, after_mean={mean_after:.4f}, "
        f"ratio={mean_after/src_valid_mean:.4f}"
    )
    return dst_array


relative_deprevation_PA = resample_avg_to_ref(
    relative_deprevation, ref_path_reprojected, relative_deprevation_path_reprojected
)

with rasterio.open(pop_bondarenko_path) as src:
    pop_bondarenko = src.read(1)

/Users/carinamanitius/Documents/GitHub/AgDownscaling/Data/Raw/Predictors/Relative_deprivation_CEISIN/povmap-grdi-v1.tif (band 1): before_mean=65.6411, after_mean=68.3255, ratio=1.0409


In [4]:
##### Step 1: country-level percentiles

def value_percentile_raster(pop_arr, value_arr, ref_path, countries_gdf, out_path=None, invert=False):
    """
    Population-weighted, within-country percentile of `value_arr`.

    - Pixels with pop > 0 and a valid value drive the CDF (weighted by population).
    - ALL land pixels with a valid value get assigned a percentile via interpolation
      onto that CDF, including pop == 0 pixels (they just carry no weight).
    - A country with zero total population is skipped (no CDF to build) -- this is
      the one case where a pixel can be "on land" but still end up with no percentile.
    """
    with rasterio.open(ref_path) as ref:
        transform, shape, crs = ref.transform, ref.shape, ref.crs
        ref_masked = ref.read(1, masked=True)
        land_mask = ~np.ma.getmaskarray(ref_masked)

    countries_gdf = countries_gdf.to_crs(crs).reset_index(drop=True)
    countries_gdf["_cid"] = np.arange(1, len(countries_gdf) + 1)

    country_raster = rasterize(
        list(zip(countries_gdf.geometry, countries_gdf["_cid"])),
        out_shape=shape, transform=transform, fill=0, dtype="int32",
    )

    pop = pop_arr.astype(np.float64)
    value = value_arr.astype(np.float64)

    valid_val = ~np.isnan(value) & land_mask
    has_pop_weight = ~np.isnan(pop) & (pop > 0)

    percentile_raster = np.full(shape, np.nan, dtype=np.float32)

    for cid in np.unique(country_raster):
        if cid == 0:
            continue
        country_mask = country_raster == cid
        rows, cols = np.where(country_mask)
        if rows.size == 0:
            continue
        rmin, rmax, cmin, cmax = rows.min(), rows.max(), cols.min(), cols.max()

        sl = np.s_[rmin:rmax+1, cmin:cmax+1]
        sub_mask = country_mask[sl]
        sub_value = value[sl]
        sub_pop = pop[sl]
        sub_valid_val = valid_val[sl] & sub_mask
        sub_weighted = sub_valid_val & has_pop_weight[sl]

        if not sub_weighted.any():
            # no population to build a CDF from in this country -- skip
            continue

        # --- build the population-weighted CDF from pop>0 pixels only ---
        w_val = sub_value[sub_weighted]
        w_pop = sub_pop[sub_weighted]

        order = np.argsort(w_val)
        sorted_val, sorted_pop = w_val[order], w_pop[order]

        unique_vals, inverse = np.unique(sorted_val, return_inverse=True)
        pop_per_val = np.zeros(len(unique_vals))
        np.add.at(pop_per_val, inverse, sorted_pop)

        cum_pop = np.cumsum(pop_per_val)
        total_pop = cum_pop[-1]
        midpoint_share = (cum_pop - pop_per_val / 2.0) / total_pop
        pct_int = np.clip(np.ceil(midpoint_share * 100), 1, 100)

        if invert:
            pct_int = 101 - pct_int  # highest value -> 0(ish), lowest value -> 100

        # --- assign a percentile to EVERY valid-value land pixel, pop>0 or not ---
        assigned = np.interp(sub_value[sub_valid_val], unique_vals, pct_int)
        assigned = np.clip(np.round(assigned), 1, 100).astype(np.float32)

        out_sub = percentile_raster[sl]
        out_sub[sub_valid_val] = assigned
        percentile_raster[sl] = out_sub

    out_meta = dict(
        driver="GTiff", height=shape[0], width=shape[1], count=1,
        dtype="float32", crs=crs, transform=transform, nodata=np.nan,
    )
    if out_path:
        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(percentile_raster, 1)

    return percentile_raster


rel_dep_pctl = value_percentile_raster(
    pop_bondarenko,
    relative_deprevation_PA,
    ref_path_reprojected,
    countries,
    out_path=rd_percentiles_reprojected,
    invert=True,
)

In [5]:
##### Step 2: global percentiles (same land / weighting / assignment logic as above)

def value_percentile_raster_global(value_arr, pop_arr, ref_path, out_path=None, invert=False):
    with rasterio.open(ref_path) as ref:
        transform, shape, crs = ref.transform, ref.shape, ref.crs
        ref_masked = ref.read(1, masked=True)
        land_mask = ~np.ma.getmaskarray(ref_masked)

    value = value_arr.astype(np.float64)
    pop = pop_arr.astype(np.float64)

    valid_val = ~np.isnan(value) & land_mask
    has_pop_weight = ~np.isnan(pop) & (pop > 0)
    weighted = valid_val & has_pop_weight

    w_val = value[weighted]
    w_pop = pop[weighted]

    order = np.argsort(w_val)
    sorted_val, sorted_pop = w_val[order], w_pop[order]

    unique_vals, inverse = np.unique(sorted_val, return_inverse=True)
    pop_per_val = np.zeros(len(unique_vals))
    np.add.at(pop_per_val, inverse, sorted_pop)

    cum_pop = np.cumsum(pop_per_val)
    total_pop = cum_pop[-1]
    midpoint_share = (cum_pop - pop_per_val / 2.0) / total_pop
    pct_int = np.clip(np.ceil(midpoint_share * 100), 1, 100)

    if invert:
        pct_int = 101 - pct_int

    # assign to every valid-value land pixel globally, not just pop>0 ones
    assigned = np.interp(value[valid_val], unique_vals, pct_int)
    assigned = np.clip(np.round(assigned), 1, 100).astype(np.float32)

    percentile_raster = np.full(shape, np.nan, dtype=np.float32)
    percentile_raster[valid_val] = assigned

    if out_path:
        out_meta = dict(
            driver="GTiff", height=shape[0], width=shape[1], count=1,
            dtype="float32", crs=crs, transform=transform, nodata=np.nan,
        )
        with rasterio.open(out_path, "w", **out_meta) as dst:
            dst.write(percentile_raster, 1)

    return percentile_raster


global_rel_dep_pctl = value_percentile_raster_global(
    relative_deprevation_PA,
    pop_bondarenko,
    ref_path_reprojected,
    out_path=rd_global_percentiles_reprojected,
    invert=True,
)